# WESAD STAI Comparison Utilities

This notebook parses the shortened six-item WESAD STAI questionnaire, applies reverse-scoring to items 1, 4, and 6, and counts how many participants have a valid TSST/STAI score equal to or higher than their score in each reference condition: Base, Fun, Medi 1, and Medi 2.

A comparison is counted only when both the TSST score and the reference-condition score contain at least one valid response.


In [9]:
from pathlib import Path
import numpy as np
import pandas as pd

REPO_ROOT = next((candidate for candidate in [Path.cwd().resolve(), Path.cwd().resolve().parent, Path.cwd().resolve().parent.parent] if (candidate / "WESAD").exists()), Path.cwd().resolve())
WESAD_DIR = REPO_ROOT / "WESAD"
OUTPUT_DIR = REPO_ROOT / "Data" / "Analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# WESAD shortened STAI order from wesad_readme.pdf.
# Positive statements are reverse-scored: 1 <-> 4 and 2 <-> 3.
STAI_ITEM_COUNT = 6
REVERSE_ITEM_INDICES = {0, 3, 5}
REFERENCE_CONDITIONS = ("Base", "Fun", "Medi 1", "Medi 2")
TARGET_CONDITION = "TSST"


In [10]:
def parse_numeric_fields(parts):
    values = []
    for part in parts:
        text = part.strip()
        if not text:
            values.append(np.nan)
            continue
        try:
            values.append(float(text))
        except ValueError:
            values.append(np.nan)
    return values


def score_stai(raw_values):
    """Return raw values and WESAD reverse-scored values."""
    raw = [float(value) if pd.notna(value) else np.nan for value in raw_values]
    if len(raw) > STAI_ITEM_COUNT:
        raise ValueError(f"Expected at most {STAI_ITEM_COUNT} STAI items, got {len(raw)}")
    if any(pd.notna(value) and not 1 <= value <= 4 for value in raw):
        raise ValueError("STAI responses must be in the range 1-4")
    scored = [
        5.0 - value if index in REVERSE_ITEM_INDICES and pd.notna(value) else value
        for index, value in enumerate(raw)
    ]
    return raw, scored


def parse_subject_quest(subject_id):
    path = WESAD_DIR / f"S{subject_id}" / f"S{subject_id}_quest.csv"
    order = None
    stai_rows = []
    with path.open("r", encoding="latin1") as handle:
        for line in handle:
            parts = [part.strip() for part in line.rstrip("\r\n").split(";")]
            tag = parts[0] if parts else ""
            if tag.startswith("# ORDER"):
                order = [value.strip() for value in parts[1:]]
            elif tag.startswith("# STAI"):
                stai_rows.append(parse_numeric_fields(parts[1:]))
    if order is None or not stai_rows:
        raise ValueError(f"{path.name}: missing ORDER or STAI rows")

    rows = []
    for index, condition in enumerate(order):
        if condition not in (*REFERENCE_CONDITIONS, TARGET_CONDITION):
            continue
        raw = [row[index] if index < len(row) else np.nan for row in stai_rows]
        raw, scored = score_stai(raw)
        valid = [value for value in scored if pd.notna(value)]
        rows.append({
            "subject_id": subject_id,
            "condition": condition,
            "stai_score": float(np.mean(valid)) if valid else np.nan,
            "valid_item_count": len(valid),
            "raw_responses": raw,
            "scored_responses": scored,
        })
    return rows


def load_scores():
    subject_ids = sorted(
        int(path.parent.name[1:])
        for path in WESAD_DIR.glob("S*/S*_quest.csv")
    )
    rows = [
        row for subject_id in subject_ids for row in parse_subject_quest(subject_id)
    ]
    return pd.DataFrame(rows)

scores_long = load_scores()
scores_long[["subject_id", "condition", "stai_score", "valid_item_count"]]


,subject_id,condition,stai_score,valid_item_count
0,2,Base,2.6,5
1,2,TSST,2.4,5
2,2,Medi 1,2.4,5
3,2,Fun,2.4,5
4,2,Medi 2,2.6,5
...,...,...,...,...
70,17,Base,3.0,5
71,17,Fun,1.8,5
72,17,Medi 1,1.8,5
73,17,TSST,3.2,5


In [11]:
def count_target_at_least_reference(scores, target=TARGET_CONDITION):
    wide = scores.pivot(index="subject_id", columns="condition", values="stai_score")
    counts = []
    for reference in REFERENCE_CONDITIONS:
        comparable = wide[[target, reference]].dropna()
        equal = comparable[target] == comparable[reference]
        higher = comparable[target] > comparable[reference]
        target_at_least = equal | higher
        counts.append({
            "target_condition": target,
            "reference_condition": reference,
            "comparable_subjects": len(comparable),
            "target_equal_count": int(equal.sum()),
            "target_equal_percent": float(equal.mean() * 100) if len(comparable) else np.nan,
            "target_higher_count": int(higher.sum()),
            "target_higher_percent": float(higher.mean() * 100) if len(comparable) else np.nan,
            "target_equal_or_higher_count": int(target_at_least.sum()),
            "target_equal_or_higher_percent": float(target_at_least.mean() * 100) if len(comparable) else np.nan,
            "target_lower_count": int((~target_at_least).sum()),
            "target_lower_percent": float((~target_at_least).mean() * 100) if len(comparable) else np.nan,
            "subject_ids_equal": comparable.index[equal].tolist(),
            "subject_ids_higher": comparable.index[higher].tolist(),
            "subject_ids_equal_or_higher": comparable.index[target_at_least].tolist(),
        })
    return pd.DataFrame(counts)

comparison_summary = count_target_at_least_reference(scores_long)
comparison_summary


,target_condition,reference_condition,comparable_subjects,target_equal_count,target_equal_percent,target_higher_count,target_higher_percent,target_equal_or_higher_count,target_equal_or_higher_percent,target_lower_count,target_lower_percent,subject_ids_equal,subject_ids_higher,subject_ids_equal_or_higher
0,TSST,Base,15,3,20.000000,6,40.000000,9,60.000000,6,40.000000,"[5, 6, 16]","[3, 4, 7, 11, 14, 17]","[3, 4, 5, 6, 7, 11, 14, 16, 17]"
1,TSST,Fun,15,3,20.000000,11,73.333333,14,93.333333,1,6.666667,"[2, 6, 16]","[3, 4, 5, 7, 8, 10, 11, 13, 14, 15, 17]","[2, 3, 4, 5, 6, 7, 8, 10, 11, 13, 14, 15, 16, 17]"
2,TSST,Medi 1,15,1,6.666667,13,86.666667,14,93.333333,1,6.666667,[2],"[3, 4, 5, 6, 7, 8, 10, 11, 13, 14, 15, 16, 17]","[2, 3, 4, 5, 6, 7, 8, 10, 11, 13, 14, 15, 16, 17]"
3,TSST,Medi 2,15,3,20.000000,11,73.333333,14,93.333333,1,6.666667,"[4, 9, 16]","[3, 5, 6, 7, 8, 10, 11, 13, 14, 15, 17]","[3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17]"


In [12]:
# Save both the corrected participant scores and the comparison counts.
scores_export = scores_long.drop(columns=["raw_responses", "scored_responses"])
scores_export.to_csv(OUTPUT_DIR / "stai_scored_condition_scores.csv", index=False)
comparison_summary.to_csv(OUTPUT_DIR / "stai_tsst_at_least_reference_counts.csv", index=False)

print("TSST STAI comparison preview (counts and percentages of valid subjects):")
preview_columns = [
    "target_condition", "reference_condition", "comparable_subjects",
    "target_equal_count", "target_equal_percent",
    "target_higher_count", "target_higher_percent",
    "target_equal_or_higher_count", "target_equal_or_higher_percent",
    "target_lower_count", "target_lower_percent",
]
preview = comparison_summary[preview_columns].copy()
percent_columns = [column for column in preview.columns if column.endswith("_percent")]
preview[percent_columns] = preview[percent_columns].round(2)
display(preview)
print(f"\nSaved results to: {OUTPUT_DIR}")

# Read the CSV outputs back and display them as notebook tables for convenient inspection.
print("\nSaved scored condition results:")
saved_scores = pd.read_csv(OUTPUT_DIR / "stai_scored_condition_scores.csv")
display(saved_scores)

print("\nSaved TSST comparison results:")
saved_comparison = pd.read_csv(OUTPUT_DIR / "stai_tsst_at_least_reference_counts.csv")
saved_comparison[percent_columns] = saved_comparison[percent_columns].round(2)
display(saved_comparison)


TSST STAI comparison preview (counts and percentages of valid subjects):


,target_condition,reference_condition,comparable_subjects,target_equal_count,target_equal_percent,target_higher_count,target_higher_percent,target_equal_or_higher_count,target_equal_or_higher_percent,target_lower_count,target_lower_percent
0,TSST,Base,15,3,20.00,6,40.00,9,60.00,6,40.00
1,TSST,Fun,15,3,20.00,11,73.33,14,93.33,1,6.67
2,TSST,Medi 1,15,1,6.67,13,86.67,14,93.33,1,6.67
3,TSST,Medi 2,15,3,20.00,11,73.33,14,93.33,1,6.67



Saved results to: E:\Rizqy\Kuliah\Skripsi\repo-skripsi\Data\Analysis

Saved scored condition results:


,subject_id,condition,stai_score,valid_item_count
0,2,Base,2.6,5
1,2,TSST,2.4,5
2,2,Medi 1,2.4,5
3,2,Fun,2.4,5
4,2,Medi 2,2.6,5
...,...,...,...,...
70,17,Base,3.0,5
71,17,Fun,1.8,5
72,17,Medi 1,1.8,5
73,17,TSST,3.2,5



Saved TSST comparison results:


,target_condition,reference_condition,comparable_subjects,target_equal_count,target_equal_percent,target_higher_count,target_higher_percent,target_equal_or_higher_count,target_equal_or_higher_percent,target_lower_count,target_lower_percent,subject_ids_equal,subject_ids_higher,subject_ids_equal_or_higher
0,TSST,Base,15,3,20.00,6,40.00,9,60.00,6,40.00,"[5, 6, 16]","[3, 4, 7, 11, 14, 17]","[3, 4, 5, 6, 7, 11, 14, 16, 17]"
1,TSST,Fun,15,3,20.00,11,73.33,14,93.33,1,6.67,"[2, 6, 16]","[3, 4, 5, 7, 8, 10, 11, 13, 14, 15, 17]","[2, 3, 4, 5, 6, 7, 8, 10, 11, 13, 14, 15, 16, 17]"
2,TSST,Medi 1,15,1,6.67,13,86.67,14,93.33,1,6.67,[2],"[3, 4, 5, 6, 7, 8, 10, 11, 13, 14, 15, 16, 17]","[2, 3, 4, 5, 6, 7, 8, 10, 11, 13, 14, 15, 16, 17]"
3,TSST,Medi 2,15,3,20.00,11,73.33,14,93.33,1,6.67,"[4, 9, 16]","[3, 5, 6, 7, 8, 10, 11, 13, 14, 15, 17]","[3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17]"
